# Configure Diagnostic Settings for Microsoft Foundry

Diagnostic settings route **platform-level logs** from your Azure AI Services resource to a Log Analytics workspace. This is the infrastructure monitoring layer — it captures API request logs, audit events, and platform metrics at the Azure resource level.

> **How this differs from Application Insights:** Diagnostic settings capture what the Azure *platform* sees (API calls, throttling, content filtering decisions). Application Insights captures what your *application code* sees (agent requests, model call dependencies, traces, exceptions). Both feed into Log Analytics, but they serve different monitoring purposes.

**What you will learn:**
- Create a Log Analytics workspace for Foundry telemetry
- Identify the available diagnostic log categories for Azure AI Services
- Configure diagnostic settings to route all platform logs and metrics
- Verify that the diagnostic pipeline is active

**Prerequisites:**
- Azure CLI installed and signed in (`az login`)
- An existing Azure AI Services resource (created when you deployed your Foundry project)
- **Contributor** role on the resource group

In [ ]:
%pip install python-dotenv -q

## Configuration

Copy `.env.example` to `.env` and fill in your values:

| Variable | Where to find it |
|---|---|
| `RESOURCE_GROUP` | Azure Portal → Resource groups |
| `AI_SERVICES_RESOURCE_NAME` | Azure Portal → your Foundry Hub → Connected resources → AI Services resource name |
| `LOG_ANALYTICS_WORKSPACE_NAME` | Name for your workspace (created if it does not exist) |
| `LOCATION` | Azure region matching your Foundry resources (e.g., `eastus2`) |

In [ ]:
import os
import subprocess
import json
from dotenv import load_dotenv

load_dotenv(override=True)

RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
AI_SERVICES_NAME = os.getenv("AI_SERVICES_RESOURCE_NAME")
WORKSPACE_NAME = os.getenv("LOG_ANALYTICS_WORKSPACE_NAME")
LOCATION = os.getenv("LOCATION", "eastus2")

# Verify Azure CLI login
result = subprocess.run(["az", "account", "show", "-o", "json"],
                        capture_output=True, text=True)
if result.returncode == 0:
    acct = json.loads(result.stdout)
    print(f"\u2705 Signed in as {acct['user']['name']}")
    print(f"   Subscription: {acct['name']}")
else:
    print("\u274c Not signed in. Run 'az login' in your terminal first.")

print(f"\nResource Group:  {RESOURCE_GROUP}")
print(f"AI Services:     {AI_SERVICES_NAME}")
print(f"Workspace:       {WORKSPACE_NAME}")
print(f"Location:        {LOCATION}")

## Step 1: Create a Log Analytics Workspace

A Log Analytics workspace is the destination for your diagnostic logs. All your KQL monitoring queries run against this workspace.

If the workspace already exists (e.g., from a previous setup), this cell displays its details. If not, it creates a new one.

In [ ]:
result = subprocess.run(
    ["az", "monitor", "log-analytics", "workspace", "show",
     "--resource-group", RESOURCE_GROUP,
     "--workspace-name", WORKSPACE_NAME,
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    workspace = json.loads(result.stdout)
    print(f"\u2705 Workspace already exists: {workspace['name']}")
else:
    print(f"Creating workspace '{WORKSPACE_NAME}' in {LOCATION}...")
    result = subprocess.run(
        ["az", "monitor", "log-analytics", "workspace", "create",
         "--resource-group", RESOURCE_GROUP,
         "--workspace-name", WORKSPACE_NAME,
         "--location", LOCATION,
         "-o", "json"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        workspace = json.loads(result.stdout)
        print(f"\u2705 Created workspace: {workspace['name']}")
    else:
        raise RuntimeError(f"Failed to create workspace:\n{result.stderr}")

WORKSPACE_ID = workspace["id"]
print(f"\nWorkspace ID:\n  {WORKSPACE_ID}")

## Step 2: Identify the AI Services Resource

Your Foundry project runs on an Azure AI Services resource (formerly Cognitive Services). This is the resource we attach diagnostic settings to — every model call, content filter decision, and API request flows through it.

In [ ]:
result = subprocess.run(
    ["az", "cognitiveservices", "account", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", AI_SERVICES_NAME,
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    ai_resource = json.loads(result.stdout)
    RESOURCE_ID = ai_resource["id"]
    print(f"\u2705 Found AI Services resource: {ai_resource['name']}")
    print(f"   Kind:     {ai_resource.get('kind', 'N/A')}")
    print(f"   Location: {ai_resource['location']}")
    print(f"\nResource ID:\n  {RESOURCE_ID}")
else:
    print(f"\u274c Could not find AI Services resource '{AI_SERVICES_NAME}'")
    print(f"\nCheck the resource name in the Azure portal:")
    print("  ai.azure.com > your project > Management > Connected resources")
    raise RuntimeError(f"AI Services resource '{AI_SERVICES_NAME}' not found in resource group '{RESOURCE_GROUP}'")

## Step 3: View Available Log Categories

Before creating the diagnostic setting, check what log categories the AI Services resource exposes. Each category represents a different type of platform data — request logs, audit events, content filtering results, etc.

In [ ]:
result = subprocess.run(
    ["az", "monitor", "diagnostic-settings", "categories", "list",
     "--resource", RESOURCE_ID,
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    categories = json.loads(result.stdout)
    print("Available diagnostic categories:\n")
    for cat in categories.get("value", []):
        cat_type = cat.get("categoryType", "unknown")
        name = cat["name"]
        print(f"  [{cat_type.upper():7s}]  {name}")
else:
    print(f"Could not retrieve categories:\n{result.stderr}")

## Step 4: Create the Diagnostic Setting

This creates a diagnostic setting that routes **all logs and metrics** to your Log Analytics workspace.

Using `categoryGroup: allLogs` captures every log category the resource supports. In production, you might selectively enable only the categories you need to control ingestion costs — but for learning, capturing everything gives you the full picture.

In [ ]:
DIAGNOSTIC_SETTING_NAME = "foundry-platform-logs"

result = subprocess.run(
    ["az", "monitor", "diagnostic-settings", "create",
     "--name", DIAGNOSTIC_SETTING_NAME,
     "--resource", RESOURCE_ID,
     "--workspace", WORKSPACE_ID,
     "--logs", '[{"categoryGroup": "allLogs", "enabled": true}]',
     "--metrics", '[{"category": "AllMetrics", "enabled": true}]',
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    setting = json.loads(result.stdout)
    print(f"\u2705 Diagnostic setting created: {setting['name']}")

    print(f"\nLog categories enabled:")
    for log in setting.get("logs", []):
        status = "\u2705" if log.get("enabled") else "\u274c"
        name = log.get("categoryGroup") or log.get("category", "unknown")
        print(f"  {status} {name}")

    print(f"\nMetrics enabled:")
    for metric in setting.get("metrics", []):
        status = "\u2705" if metric.get("enabled") else "\u274c"
        print(f"  {status} {metric.get('category', 'unknown')}")
else:
    print(f"\u274c Failed to create diagnostic setting:\n{result.stderr}")

## Step 5: Verify the Configuration

List all diagnostic settings on the AI Services resource to confirm everything is wired up.

In [ ]:
result = subprocess.run(
    ["az", "monitor", "diagnostic-settings", "list",
     "--resource", RESOURCE_ID,
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    settings = json.loads(result.stdout)
    items = settings.get("value", settings) if isinstance(settings, dict) else settings
    print(f"Diagnostic settings for {AI_SERVICES_NAME}:\n")
    for s in items:
        print(f"  Name:        {s['name']}")
        destinations = []
        if s.get("workspaceId"):
            destinations.append("Log Analytics")
        if s.get("storageAccountId"):
            destinations.append("Storage Account")
        if s.get("eventHubAuthorizationRuleId"):
            destinations.append("Event Hub")
        print(f"  Destination: {', '.join(destinations)}")
        log_count = sum(1 for l in s.get("logs", []) if l.get("enabled"))
        metric_count = sum(1 for m in s.get("metrics", []) if m.get("enabled"))
        print(f"  Logs:        {log_count} category/group(s) enabled")
        print(f"  Metrics:     {metric_count} category/group(s) enabled")
        print()
else:
    print(f"\u274c Could not list diagnostic settings:\n{result.stderr}")

## Summary

You configured diagnostic settings to route platform logs from your Azure AI Services resource to a Log Analytics workspace.

**What is flowing now:**
- **Request logs** — every API call to your AI Services resource (completions, embeddings, content filtering)
- **Audit logs** — configuration changes, deployment operations, access events
- **Platform metrics** — request counts, latency, throttled requests, token transaction counts

**Where to find the data:**
- Azure Portal → your Log Analytics workspace → Logs
- Query the `AzureDiagnostics` table or resource-specific tables
- Platform metrics appear in `AzureMetrics`

**Application-level telemetry** — agent requests, model call dependencies, and traces — is captured by Application Insights. That is where the `AppRequests`, `AppDependencies`, and `AppTraces` tables used in the KQL demos get their data.